In [1]:
from transformer_lens import HookedTransformer as ht
model = ht.from_pretrained("gpt2-small")

/var/folders/tq/t53mr3x93871q_pjlj_254n40000gn/T/ipykernel_18033/753418759.py:2: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = ht.from_pretrained("gpt2-small")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [24]:
tokens = model.to_tokens("a dog and a cat")
logits, cache = model.run_with_cache(tokens)

In [25]:
cache.keys()

dict_keys(['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.hook_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resid_post', 'blocks.2.hook_resid_pre', 'block

In [26]:
pattern = cache['pattern',0]
print(pattern.shape)

torch.Size([1, 12, 6, 6])


In [27]:
tokens = model.to_str_tokens("a dog and a cat")
_, cache = model.run_with_cache('a dog and a cat')
pattern = cache["pattern", 0][0]  # [n_heads, seq, seq]

for head in range(pattern.shape[0]):
    print(f"\nHead {head}")
    print("tokens:", tokens)
    for row in pattern[head]:
        print([round(x.item(), 2) for x in row])


Head 0
tokens: ['<|endoftext|>', 'a', ' dog', ' and', ' a', ' cat']
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.97, 0.03, 0.0, 0.0, 0.0, 0.0]
[0.79, 0.07, 0.14, 0.0, 0.0, 0.0]
[0.72, 0.07, 0.17, 0.04, 0.0, 0.0]
[0.7, 0.05, 0.19, 0.03, 0.03, 0.0]
[0.47, 0.07, 0.06, 0.14, 0.08, 0.17]

Head 1
tokens: ['<|endoftext|>', 'a', ' dog', ' and', ' a', ' cat']
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
[0.0, 0.15, 0.0, 0.0, 0.84, 0.0]
[0.0, 0.0, 0.01, 0.0, 0.0, 0.99]

Head 2
tokens: ['<|endoftext|>', 'a', ' dog', ' and', ' a', ' cat']
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.96, 0.04, 0.0, 0.0, 0.0, 0.0]
[0.8, 0.11, 0.09, 0.0, 0.0, 0.0]
[0.51, 0.05, 0.05, 0.39, 0.0, 0.0]
[0.62, 0.07, 0.09, 0.17, 0.04, 0.0]
[0.68, 0.09, 0.04, 0.05, 0.09, 0.05]

Head 3
tokens: ['<|endoftext|>', 'a', ' dog', ' and', ' a', ' cat']
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.23, 0.77, 0.0, 0.0, 0.0, 0.0]
[0.03, 0.01, 0.96, 0.0, 0.0, 0.0]
[0.15, 0.07, 0.06, 0.71, 0

In [31]:
def zero_head_6(pattern, hook):
    pattern[:, 6, :, :] = 0.0
    return pattern

logits_ablated = model.run_with_hooks(
    'a dog and a cat',
    fwd_hooks=[("blocks.0.attn.hook_pattern", zero_head_6)]
)
logits_normal = model('a dog and a cat')

next_normal = logits_normal[0, -1].argmax().item()
next_ablated = logits_ablated[0, -1].argmax().item()

print("normal prediction:", repr(model.to_single_str_token(next_normal)))
print("ablated prediction:", repr(model.to_single_str_token(next_ablated)))
probs_normal = logits_normal[0, -1].softmax(dim=-1)
probs_ablated = logits_ablated[0, -1].softmax(dim=-1)

print("normal probability:", probs_normal[next_normal].item())
print("ablated probability:", probs_ablated[next_normal].item())

normal prediction: '\n'
ablated prediction: '\n'
normal probability: 0.14367806911468506
ablated probability: 0.150508850812912


So i checked each head's attention pattern after softmax and saw that each head mainly had a pattern according to the sentence input. Head 6 however, did not change the distribution of the weights based on input, telling that its weights are based on the position and not what is at that position. i ablated the head 6 and zeroed out its attention pattern from layer 0 since even if a head is paying attention to something it can still be useless for the final output, maybe because of how proj or next layer assigns importance to it. after zeroing out, prediction was still the same and probability barely moved, if anything it went up slightly. However this is not enough to say that head 6 is useless in general because we only ablated 1 head for one sentence and one query position and only tested using last token to predict next.